In [19]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("okienka").getOrCreate()
spark

In [20]:
czujnik_temperatury = ((12.5, "2019-01-02 12:00:00"),
(17.6, "2019-01-02 12:00:20"),
(14.6,  "2019-01-02 12:00:30"),
(22.9,  "2019-01-02 12:01:15"),
(17.4,  "2019-01-02 12:01:30"),
(25.8,  "2019-01-02 12:03:25"),
(27.1,  "2019-01-02 12:02:40"),
)

In [21]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("temperatura", DoubleType(), True),
    StructField("czas", StringType(), True),
])

In [22]:
df = (spark.createDataFrame(czujnik_temperatury, schema=schema)
      .withColumn("czas", to_timestamp("czas")))

df.printSchema()

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [23]:
df.show(3)

+-----------+-------------------+
|temperatura|               czas|
+-----------+-------------------+
|       12.5|2019-01-02 12:00:00|
|       17.6|2019-01-02 12:00:20|
|       14.6|2019-01-02 12:00:30|
|       22.9|2019-01-02 12:01:15|
|       17.4|2019-01-02 12:01:30|
|       25.8|2019-01-02 12:03:25|
|       27.1|2019-01-02 12:02:40|
+-----------+-------------------+



In [13]:
df.createOrReplaceTempView("df")

In [14]:
spark.sql("select czas, temperatura from df where temperatura > 21").show(5)

+-------------------+-----------+
|               czas|temperatura|
+-------------------+-----------+
|2019-01-02 12:01:15|       22.9|
|2019-01-02 12:03:25|       25.8|
|2019-01-02 12:02:40|       27.1|
+-------------------+-----------+



In [25]:
# Thumbling window

import pyspark.sql.functions as F

df2 = df.groupBy(F.window("czas","30 seconds")).count()
df2.show(truncate=False)

+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2019-01-02 12:00:00, 2019-01-02 12:05:00}|7    |
+------------------------------------------+-----+



In [16]:
df2.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- count: long (nullable = false)



In [28]:
spark.stop()

In [ ]:
df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

In [29]:
%%file streamrate.py
## uruchom przez spark-submit streamrate.py

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)


query = (df.writeStream 
    .format("console") 
    .outputMode("append") 
    .option("truncate", False) 
    .start()
) 

query.awaitTermination()

Writing streamrate.py


In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()


df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+

Batch ID: 1
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 07:01:07.332|0    |
+-----------------------+-----+

Batch ID: 2
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 07:01:08.332|1    |
+-----------------------+-----+

Batch ID: 3
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 07:01:09.332|2    |
+-----------------------+-----+

Batch ID: 4
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 07:01:10.332|3    |
+-----------------------+-----+

Batch ID: 5
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 07:01:11.332|4    |
+-----------------------+-----+

Batch ID: 0
+----+-----------+
|czas|temperatura|


In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

query = (stream.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_filtered = stream.filter(col("temperatura") > 28)

query = (stream_filtered.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, when

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=10):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_anomaly = stream.withColumn(
    "anomaly",
    when(col("temperatura") > 29.5, "TAK").otherwise("NIE")
)

query = (stream_anomaly.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+----+-----------+-------+
|czas|temperatura|anomaly|
+----+-----------+-------+
+----+-----------+-------+

Batch ID: 1
+-----------------------+-----------------+-------+
|czas                   |temperatura      |anomaly|
+-----------------------+-----------------+-------+
|2026-04-20 07:09:12.911|26.05877148559475|NIE    |
+-----------------------+-----------------+-------+

Batch ID: 2
+-----------------------+------------------+-------+
|czas                   |temperatura       |anomaly|
+-----------------------+------------------+-------+
|2026-04-20 07:09:13.911|29.184694354779392|NIE    |
+-----------------------+------------------+-------+

Batch ID: 3
+-----------------------+------------------+-------+
|czas                   |temperatura       |anomaly|
+-----------------------+------------------+-------+
|2026-04-20 07:09:14.911|20.348155671060425|NIE    |
+-----------------------+------------------+-------+

Batch ID: 4
+-----------------------+-------------

In [4]:
from pyspark.sql.functions import col, expr
from pyspark.sql import SparkSession
from pyspark.sql.functions import window

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=30):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )
# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))

tumbling_window = (stream
    .groupBy(window(col("czas"), "10 seconds"))
    .avg("temperatura")
)

query = (tumbling_window.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+------+----------------+
|window|avg(temperatura)|
+------+----------------+
+------+----------------+

Batch ID: 1
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-20 07:17:40, 2026-04-20 07:17:50}|27.08394945179064 |
|{2026-04-20 07:17:30, 2026-04-20 07:17:40}|23.022875650757648|
+------------------------------------------+------------------+

Batch ID: 2
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-20 07:17:40, 2026-04-20 07:17:50}|26.52247344803707 |
|{2026-04-20 07:17:50, 2026-04-20 07:18:00}|28.104823528831176|
|{2026-04-20 07:17:30, 2026-04-20 07:17:40}|23.022875650757648|
+------------------------------------------+------------------+

Batch ID: 3


In [2]:
from pyspark.sql.functions import col, expr
from pyspark.sql import SparkSession
from pyspark.sql.functions import window

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=30):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )
# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))


sliding = (stream.groupBy(window(col("czas"), "10 seconds", "5 seconds"))
           .avg("temperatura")
          )

query = (sliding.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+------+----------------+
|window|avg(temperatura)|
+------+----------------+
+------+----------------+

Batch ID: 1
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-20 07:18:35, 2026-04-20 07:18:45}|22.864283206401442|
|{2026-04-20 07:18:25, 2026-04-20 07:18:35}|25.554170611055746|
|{2026-04-20 07:18:30, 2026-04-20 07:18:40}|24.209226908728592|
+------------------------------------------+------------------+

Batch ID: 2
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-20 07:18:40, 2026-04-20 07:18:50}|24.52022069870469 |
|{2026-04-20 07:18:35, 2026-04-20 07:18:45}|23.692251952553068|
|{2026-04-20 07:18:25, 2026-04-20 07:18:35}|25.554170611055746|
|{2026-04-20 07:18:30, 202

In [3]:
%%file generator.py
# generator.py
import json, os, random, time
from datetime import datetime, timedelta

output_dir = "data/stream"
os.makedirs(output_dir, exist_ok=True)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

# Simulate file-based streaming
while True:
    batch = [generate_transaction() for _ in range(2)]
    filename = f"{output_dir}/events_{int(time.time())}.json"
    with open(filename, "w") as f:
        for e in batch:
            f.write(json.dumps(e) + "\n")
    print(f"Wrote: {filename}")
    time.sleep(5)

Writing generator.py


In [1]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, to_timestamp, sum, count
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("jsonDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

batch_counter = {"count": 0}

def process_batch(df, batch_id, tstop=5):
    batch_counter["count"] += 1
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
    


stream = (spark.readStream
          .schema(tx_schema)
          .json("data/stream"))

aggregated_stream = (stream
    # Konwersja czasu ze String na Timestamp
    .withColumn("timestamp", to_timestamp(col("timestamp"))) 
    # Grupowanie i obliczenia
    .groupBy("category")
    .agg(
        sum("amount").alias("suma_wydatkow"),
        count("tx_id").alias("liczba_transakcji")
    )
)

# 2. Uruchomienie zapytania z trybem "complete"
query = (aggregated_stream.writeStream
         .outputMode("complete")  # Tryb complete jest wymagany przy agregacjach
         .foreachBatch(process_batch)
         .start())
query = (stream.writeStream
         .format("console")
         .foreachBatch(process_batch)
         .start())

TypeError: unsupported operand type(s) for +: 'int' and 'str'